In [1]:
library(Seurat)
source('plotDesign.r')
source('FNC.r')
sfile <- function(filename, folder="output") FNC(folder, filename, pfx="1_annot_coi", sep="_", pfxDate = FALSE)

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t




# Altas Filtering

<b> Information </b>

<ol>
<li>
The atlas rds file metadata is without annotation, please find atlas annotation from the csv file.
</li><li>
For CD4 analysis, sample A05 and NUC010A were removed. Because CD4 from these two samples are less than 100 cells which strongly influence integration quality. Therefore, there'll be annotation drop-off in atlas CD4 subclusters. Just ignore CD4 from these two samples.
</li><li>
For CD8, TEM-a,b,c,d,e,f,g together are auto-aggressive-like CD8.
</li><li>
For hepatocytes, there're two clusters named PTPRC-pos and Endothelial-pos. They're undefined, maybe doublets, please remove them for interactom analysis.
</li>
</ol>

In [2]:
org_seu <- readRDS("3_SCS_SNS_atlas.rds")
atlas_annotation <- read.csv("Interactom_annotation/Interactom_atlas_annotation.csv")
CD4_annotation <- read.csv("Interactom_annotation/Interactom_CD4_annotation.csv")
CD8_annotation <- read.csv("Interactom_annotation/Interactom_CD8_annotation.csv")
hepatocytes_annotation <- read.csv("Interactom_annotation/Interactom_hepatocytes_annotation.csv")

In [ ]:
org_seu@meta.d

In [3]:
# Set annotation to deepest level
deep_annotation <- atlas_annotation$annotation
names(deep_annotation) <- atlas_annotation$X
deep_annotation[CD4_annotation$X] <- paste0(deep_annotation[CD4_annotation$X],
                                            "_",
                                            CD4_annotation$annotation)
deep_annotation[CD8_annotation$X] <- paste0(deep_annotation[CD8_annotation$X],
                                            "_",
                                            CD8_annotation$annotation)
deep_annotation[hepatocytes_annotation$X] <- paste0(deep_annotation[hepatocytes_annotation$X],
                                            "_",
                                            hepatocytes_annotation$annotation)


# 1.  RM CD4 from  A05 and NUC010A
rm_CD4 <- !(grepl("A05|NUC010A",names(deep_annotation)) & grepl("CD4", deep_annotation))
message(paste0(sum(!rm_CD4), " cells will be removed, because A05 or NUC010A and CD4"))

final_annotation <- deep_annotation[rm_CD4]

# 2. For CD8, TEM-a,b,c,d,e,f,g together are auto-aggressive-like CD8.

aaCD8 <- grepl("CD8_TEM-[a-g]",final_annotation)
message("Auto Aggressive CD8 cells:")
knitr::kable(table(final_annotation[aaCD8]))
final_annotation[aaCD8] <- "aa-like CD8"

# 3. For hepatocytes, there're two clusters named PTPRC-pos and Endothelial-pos. 
# They're undefined, maybe doublets, please remove them for interactom analysis.

rm_hepa <- !grepl("Endothelial-pos|PTPRC-pos", final_annotation)
message("Undefined Hepato:")
knitr::kable(table(final_annotation[!rm_hepa]))
final_annotation <- final_annotation[rm_hepa]

# Merge all hepatocyte clusters
hepa <- grepl("Hepatocytes", final_annotation)
final_annotation[hepa] <- "Hepatocytes"

# Select only clusters of interest
coi <- c("aa-like CD8", "CD4_TRM1", "Hepatocytes", "Macrophages/DC")
final_annotation <- final_annotation[final_annotation %in% coi] 

# Final annotation
message("Final annotation:")
knitr::kable(table(final_annotation))

saveRDS(final_annotation, sfile("annotation.rds"))


127 cells will be removed, because A05 or NUC010A and CD4

Auto Aggressive CD8 cells:





|Var1      | Freq|
|:---------|----:|
|CD8_TEM-a | 3828|
|CD8_TEM-b | 5336|
|CD8_TEM-c |  408|
|CD8_TEM-d | 2817|
|CD8_TEM-e | 1902|
|CD8_TEM-f | 1659|
|CD8_TEM-g |  113|

Undefined Hepato:





|Var1                        | Freq|
|:---------------------------|----:|
|Hepatocytes_Endothelial-pos |  415|
|Hepatocytes_PTPRC-pos       | 2392|

Final annotation:





|final_annotation |  Freq|
|:----------------|-----:|
|aa-like CD8      | 16063|
|CD4_TRM1         |  1018|
|Hepatocytes      | 13307|
|Macrophages/DC   |  7225|

In [4]:
seu <- subset(org_seu, cells = names(final_annotation))
print(paste0("cells original: ",dim(org_seu)[2]))
print(paste0("cells coi: ",dim(seu)[2]))

# save filtered seu
saveRDS(seu, sfile("seu.rds"))

# to save memory
rm(seu_org)

[1] "cells original: 74102"
[1] "cells coi: 37613"


Warning message in rm(seu_org):
“object 'seu_org' not found”


In [5]:
sessionInfo()

R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 22.04.3 LTS

Matrix products: default
BLAS/LAPACK: /opt/conda/envs/r-scrna/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] Seurat_5.3.0       SeuratObject_5.2.0 sp_2.2-0          

loaded via a namespace (and not attached):
  [1] deldir_2.0-4           pbapply_1.7-4          gridExtra_2.3         
  [4] rlang_1.1.6            magrittr_2.0.4         RcppAnnoy_0.0.22      
  [7] otel_0.2.0             spatstat.geom_3.